### Module 10b — GraphRAG

In [2]:
import os
from pathlib import Path
import logging
from dotenv import load_dotenv
import networkx as nx
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

os.environ['ANONYMIZED_TELEMETRY'] = 'False' 
logging.getLogger('httpx').setLevel(logging.WARNING)

Demonstrate the bridge-question failure

In [3]:
# Load the domain-tagged vector store 
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

embeddings = OpenAIEmbeddings()
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print(f'Store loaded — {vectorstore._collection.count()} chunks')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Store loaded — 80 chunks


In [4]:
# Normal question — standard RAG 
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer using ONLY the provided context. '
               'If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {input}')
])
qa_chain = qa_prompt | llm | StrOutputParser()

def ask(question):
    docs = retriever.invoke(question)
    context = '\n\n'.join(d.page_content for d in docs)
    return qa_chain.invoke({'context': context, 'input': question})

q1 = 'What are the common crop diseases in Nigeria?'
print(f'👤 {q1}')
print(f'🤖 {ask(q1)[:300]}')
print('-' * 70)

👤 What are the common crop diseases in Nigeria?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🤖 The common crop diseases in Nigeria are:

1. Cassava Mosaic Disease
   - Affected Crop: Cassava
   - Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.

2. Maize Smut
   - Affected Crop: Maize
   - Symptoms: Large grey or black galls on ears, stalks, and leaves.

3. Rice Blas
----------------------------------------------------------------------


In [5]:
# Bridge question — asks about a relationship between two things
q2 = 'What do malaria and rice blast have in common?'
print(f'👤 {q2}')
print(f'🤖 {ask(q2)}')
print('-' * 70)

👤 What do malaria and rice blast have in common?
🤖 I don't know.
----------------------------------------------------------------------


In [6]:
# See what was retrieved for the bridge question
docs = retriever.invoke('What do malaria and rice blast have in common?')

print(f'Retrieved {len(docs)} chunks for the bridge question:\n')
for i, d in enumerate(docs):
    domain = d.metadata.get('domain', '?')
    print(f'--- Chunk {i} (domain={domain}) ---')
    print(d.page_content[:250].replace('\n', ' '))
    print()

Retrieved 4 chunks for the bridge question:

--- Chunk 0 (domain=crops) ---
Common Crop Diseases and Their Control  1. Cassava Mosaic Disease Affected Crop: Cassava Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield. Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infecte

--- Chunk 1 (domain=crops) ---
Fisheries and aquaculture provide employment and protein for many families. Catfish and tilapia are commonly farmed in ponds and tanks.  Common Crop Diseases and Control  1. Cassava Mosaic Disease    Affected crop: Cassava    Symptoms: Yellowing and 

--- Chunk 2 (domain=crops) ---
Cassava O Disease: Cassava bacterial blight O Causative organism: Xanthomonas manihotis

--- Chunk 3 (domain=crops) ---
Cassava O Disease: Cassava Mosaic Virus (CMV)  disease  O Causative organism: Trasmitted by  white fly (Bemisia tabaci)

